# 🚌 대구버스 공공데이터 API 실습 워크시트

**목표:** 공공데이터포털 API 문서를 읽고, 직접 데이터를 불러와봅니다.

---
## ✅ 시작 전 체크리스트
- [ ] 공공데이터포털 회원가입 완료
- [ ] 대구버스정보시스템 API 활용 신청 완료
- [ ] API 키 승인 완료 (마이페이지 > 활용신청 현황에서 확인)

> 📌 API 페이지 주소: https://www.data.go.kr/data/15139134/openapi.do

---
## 📖 Step 0. API가 뭔가요?

**API(Application Programming Interface)** 는 프로그램끼리 데이터를 주고받는 약속된 방법입니다.

```
🙋 나(파이썬 코드)
      ↓ requests.get() 으로 요청
🏢 공공데이터 서버
      ↓ JSON 데이터로 응답
📦 데이터 수신!
```

### 응답 코드 의미
| 코드 | 의미 |
|------|------|
| **200** | ✅ 성공! |
| **401** | 🔴 API 키 오류 |
| **400** | 🔴 파라미터 오류 |
| **500** | 🔴 서버 오류 |

---
## 🔑 Step 1. API 키 설정

공공데이터포털 마이페이지에서 발급받은 **일반 인증키(Decoding)** 를 입력하세요.

> ⚠️ 주의: 키 앞뒤에 공백이나 탭이 들어가면 401 오류가 납니다! `.strip()` 필수!

In [31]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv('OPENAPI_API_KEY')

In [32]:
import requests
import json

print("API 키 길이:", len(API_KEY))
print("앞 10자리:", API_KEY[:10], "...")

API 키 길이: 64
앞 10자리: 0f4cc2e337 ...


---
## 📡 Step 2. API 문서 읽기 연습

아래 사이트에서 **getBasic02** 항목을 찾아 다음을 채워보세요.

🌐 https://www.data.go.kr/data/15139134/openapi.do

### 📝 탐색 미션 (빈칸 채우기)

```
1. API 이름: getBasic02
2. 기능 설명: 기초 종합 정보 조회 서비스
3. 필수 파라미터(*): serviceKey
4. 응답 데이터에 포함된 항목들:
   - route (노선): routeId, routeNo, stNm, edNm
   - bs (정류장): bsId, bsNm, xPos, yPos
   - node (노드): nodeId, nodeNm, xPos, bsYn
```

---
## 🚀 Step 3. getBasic02 호출하기 (전체 기초 데이터)

In [18]:
# 📡 API 주소
url = "https://apis.data.go.kr/6270000/dbmsapi02/getBasic02"

# 📋 요청 파라미터
params = {
    "serviceKey": API_KEY
}

# 🚀 요청 보내기
response = requests.get(url, params=params)

# ✅ 결과 확인
print("응답 코드:", response.status_code)

if response.status_code == 200:
    print("✅ 성공!")
elif response.status_code == 401:
    print("🔴 API 키를 확인하세요!")
else:
    print("🔴 오류:", response.status_code)

응답 코드: 200
✅ 성공!


---
## 🔍 Step 4. 응답 데이터 구조 탐색

In [19]:
# JSON으로 변환
data = response.json()

# 전체 키 확인
print("최상위 키:", list(data.keys()))

최상위 키: ['header', 'body']


In [20]:
# 헤더 확인 (성공 여부)
header = data["header"]
print("성공 여부:", header["success"])
print("결과 메시지:", header["resultMsg"])
print("결과 코드:", header["resultCode"])

성공 여부: True
결과 메시지: 성공
결과 코드: 0000


In [21]:
# 바디 확인 (실제 데이터)
body = data["body"]
print("총 데이터 수:", body["totalCount"])
print("items 키 목록:", list(body["items"].keys()))

총 데이터 수: 25172
items 키 목록: ['bs', 'node', 'route', 'link']


---
## 🚌 Step 5. 노선 정보 꺼내기

In [22]:
# 노선 데이터
routes = data["body"]["items"]["route"]

print(f"전체 노선 수: {len(routes)}개")
print("\n첫 번째 노선 정보:")
print(json.dumps(routes[0], ensure_ascii=False, indent=2))

전체 노선 수: 522개

첫 번째 노선 정보:
{
  "routeId": "4050003000",
  "routeNo": "동구3",
  "routeNote": "부동방면",
  "routeTCd": "4",
  "stBsId": "7021051300",
  "edBsId": "7011030000",
  "stNm": "칠성시장",
  "edNm": "부동",
  "dirRouteNote": "부동",
  "ndirRouteNote": "칠성시장",
  "dataconnareacd": "DG"
}


In [23]:
# 노선 번호만 출력 (앞 10개)
print("노선 번호 목록 (앞 10개):")
for route in routes[:10]:
    print(f"  - {route['routeNo']} ({route['stNm']} → {route['edNm']})")

노선 번호 목록 (앞 10개):
  - 동구3 (칠성시장 → 부동)
  - 504 (방천리 → 두리봉터널)
  - 232-1 (방천리 → 방천리)
  - 403 (범물동 → 조야동)
  - 405 (삼산리 → 방천리)
  - 413 (대일리 → 종합유통단지)
  - 503 (연경동 → 성서산업단지)
  - 706 (관음동 → 대곡지구)
  - 708 (동호동(북구) → 동호지구)
  - 719 (관음동 → 금구리)


### 🎯 미션 1
아래 코드를 완성해서 **급행** 노선만 출력해보세요!

In [24]:
# 🎯 미션 1: 급행 노선만 찾기
print("급행 노선 목록:")
for route in routes:
    if "급행" in route["routeNo"]:
        print(f"  🚌 {route['routeNo']} | {route['stNm']} → {route['edNm']}")

급행 노선 목록:
  🚌 급행8 | 달성2차산업단지 → 서대구역
  🚌 급행8-1 | 유곡리 → 진천역
  🚌 급행9 | 동호동 → 군위군청
  🚌 급행9-2 | 동호동 → 삼국유사면문화관
  🚌 급행2 | 냉천리 → 동호동(북구)
  🚌 급행10 | 대천동 → 반야월역
  🚌 급행9-3 | 동호동 → 소보시장
  🚌 급행9-1 | 동호동 → 우보정류장
  🚌 급행5 | 성서산업단지 → 대구대
  🚌 급행7 | 대곡동(공영차고지) → 동호동
  🚌 급행1 | 동화사 → 동대구역
  🚌 급행1 | 동화사 → 매곡
  🚌 급행3 | 범물동 → 동명면


---
## 🚏 Step 6. 정류장 정보 꺼내기

In [25]:
# 정류장 데이터
stops = data["body"]["items"]["bs"]

print(f"전체 정류장 수: {len(stops)}개")
print("\n첫 번째 정류장 정보:")
print(json.dumps(stops[0], ensure_ascii=False, indent=2))

전체 정류장 수: 5913개

첫 번째 정류장 정보:
{
  "bsId": "7041000100",
  "bsNm": "한국방송통신대학교건너",
  "xPos": 128.49331521666667,
  "yPos": 35.859160116666665,
  "wincId": "01153"
}


### 🎯 미션 2
아래 빈칸을 채워서 **"동대구역"** 이 이름에 포함된 정류장을 모두 출력해보세요!

In [26]:
# 🎯 미션 2: 동대구역 정류장 찾기
keyword = "동대구역"   

print(f"'{keyword}' 포함 정류장:")
for stop in stops:
    if keyword in stop["bsNm"]:   # ← 빈칸: 정류장 이름 필드명은?
        print(f"  🚏 {stop['bsNm']} | ID: {stop['bsId']}")

'동대구역' 포함 정류장:
  🚏 동대구역센트럴시티자이아파트 | ID: 7011003500
  🚏 동대구역복합환승센터앞 | ID: 7011004000
  🚏 동대구역복합환승센터건너 | ID: 7011004100
  🚏 동대구역 | ID: 7011006700
  🚏 동대구역건너 | ID: 7011006800
  🚏 동대구역지하도1 | ID: 7011008000
  🚏 동대구역지하도2 | ID: 7011008100
  🚏 동대구역수화물취급소건너 | ID: 7011021200
  🚏 동대구역수화물취급소앞 | ID: 7011021300
  🚏 동대구역화성파크드림 | ID: 7011006900
  🚏 동대구역(2번출구)건너 | ID: 7011067300


---
## 🔗 Step 7. 다른 API 탐색하기

API 문서 페이지에서 다른 API들을 찾아 탐색해봅시다.

### 📝 탐색 미션 (빈칸 채우기)

| API 이름 | 기능 | 필수 파라미터 | 내가 뽑고 싶은 데이터 |
|----------|------|--------------|---------------------|
| getBasic02 | 기초종합 정보 | serviceKey | 노선/정류장 목록 |
| getBs02 | 노선별 경유 정류장 정보 조회 서비스 | serviceKey, routeId | |
| getLink02 | 노선별 경유 링크 정보 조회 서비스 | serviceKey, routeId | |
| getRealtime02 | 정류소별 버스 도착예정정보 조회 서비스 | serviceKey, bsId | |
| getPos02 | 노선별 경유 버스 위치 조회 서비스 | serviceKey, routeId | |

In [30]:
# 🎯 미션 3: getBs02 - 정류장 상세 정보 조회
# Step 6에서 찾은 정류장 ID를 활용해보세요!

url_bs = "https://apis.data.go.kr/6270000/dbmsapi02/getBs02"

params_bs = {
    "serviceKey": API_KEY,
    "routeId": "7011006700"   # ← Step 6에서 찾은 bsId 입력
}

response_bs = requests.get(url_bs, params=params_bs)
print("응답 코드:", response_bs.status_code)

if response_bs.status_code == 200:
    data_bs = response_bs.json()
    print(json.dumps(data_bs, ensure_ascii=False, indent=2))

응답 코드: 200
{
  "header": {
    "success": false,
    "resultCode": "9003",
    "resultMsg": "인증키오류 및 파라미터 오류"
  }
}


---
## 🏆 Step 8. 나만의 탐색 코드 작성

지금까지 배운 것을 활용해서 **내가 원하는 정보를 조회하는 코드**를 작성해보세요!

**아이디어 예시:**
- 집 근처 정류장 찾기
- 특정 번호 버스의 출발지/도착지 확인
- 정류장 이름에 '학교'가 들어가는 곳 찾기

In [28]:
# 🏆 나만의 탐색 코드
# 아이디어: ___________________________

# 여기에 코드를 작성하세요!


---
## 📚 정리

오늘 배운 것:

| 개념 | 코드 | 설명 |
|------|------|------|
| API 요청 | `requests.get(url, params=params)` | 서버에 데이터 요청 |
| 응답 확인 | `response.status_code` | 200이면 성공 |
| JSON 변환 | `response.json()` | 딕셔너리로 변환 |
| 데이터 접근 | `data["body"]["items"]` | 키로 접근 |
| 데이터 탐색 | `for item in items:` | 반복문으로 탐색 |
| 조건 필터 | `if "키워드" in item["필드"]:` | 원하는 것만 추출 |
